# Run an experiment on a Kaggle GPU

Accelerator: GPU T4 x1 or P100. Internet: on. Inputs: the private datasets `sixth-finger-bci-epochs` and, for fine-tuning, `sixth-finger-bci-pretrained`. Add the previous version's output as an input to resume an interrupted grid.

In [ ]:
EXPERIMENT = "configs/experiments/rq2_calib_s1.yaml"
METHODS = "eegnet_scratch,eegnet_pooled,eegnet_ea_ft,eegnet_ea_head"
SUBJECTS = "all"
REPO = "https://github.com/<owner>/sixth-finger-bci.git"

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GH_TOKEN")
url = REPO.replace("https://", f"https://{token}@")
subprocess.run(["git", "clone", "--depth", "1", url, "repo"], check=True)
os.chdir("repo")

In [ ]:
# Kaggle already has a CUDA build of torch; install the rest of the pinned stack around it.
!pip install -q -e . --no-deps
!pip install -q "mne==1.13.2" "pymatreader==1.3.2" "pyriemann==0.12"
!pip install -q "braindecode==1.8.1" "pydantic>=2.13" "typer>=0.27"
!pip install -q pingouin statsmodels pyarrow
import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu"
print(torch.__version__, gpu)

In [ ]:
!mkdir -p data/derived models/pretrained results/runs
!ln -sfn /kaggle/input/sixth-finger-bci-epochs data/derived/epochs
!cp /kaggle/input/sixth-finger-bci-pretrained/*.pt models/pretrained/ 2>/dev/null || true
!cp -r /kaggle/input/*/out/runs/* results/runs/ 2>/dev/null || true

In [ ]:
!sixthfinger doctor
!sixthfinger run {EXPERIMENT} --methods {METHODS} --subjects {SUBJECTS}

In [ ]:
!mkdir -p /kaggle/working/out
!cp -r results/runs /kaggle/working/out/
!cp -r models/pretrained /kaggle/working/out/ 2>/dev/null || true
!ls -R /kaggle/working/out | head -50